In [1]:
from data_cleaning.powerbi_entry import final_df

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [9]:
state_df=final_df.groupby('state',as_index=True)[['enrol_age_0_5','enrol_age_5_17','enrol_age_18+','demo_age_5_17','demo_age_17+','bio_age_5_17','bio_age_17+',]].sum().reset_index()

In [17]:
state_df['Enrolment']=(state_df['enrol_age_0_5']+state_df['enrol_age_5_17']+state_df['enrol_age_18+'])/1000000

state_df['Biometric']=(state_df['bio_age_5_17']+state_df['bio_age_17+'])/1000000

state_df['Demographic']=(state_df['demo_age_5_17']+state_df['demo_age_17+'])/1000000
state_df['Total']=state_df['Enrolment']+state_df['Biometric']+state_df['Demographic']

In [34]:
df=state_df.copy()
df['Total_update']=df['Demographic']+df['Biometric']
df['Enrolment_update_ratio']=df['Enrolment']/df['Total_update'].replace(0,np.nan)

fig=px.scatter(
    df,
    x='Enrolment_update_ratio',
    y='Total',
    size='Total',
    color='Total',
    hover_name='state',
    color_continuous_scale='Viridis',
    labels={
        'Enrolment_update_ratio':'Enrolment Update Ratio',
        'Total':'Total in million'
    },
    title='Aadhaar Ecosystem Maturity Matrix (State-wise)'
)
fig.add_vline(
    x=df['Enrolment_update_ratio'].median(),
    line_dash='dash',
    line_color='red',
    annotation_text='Median Enrol/Update Ratio',
    annotation_position='top'
)
fig.add_hline(
    y=df['Total'].median(),
    line_dash='dash',
    line_color='blue',
    annotation_text='Median Activity',
    annotation_position='right')
fig.update_layout(
    height=650,
    template='plotly_white'
)
fig.show()

In [35]:
mean_update_ratio=df['Enrolment_update_ratio'].mean(skipna=True)
std_ratio=df['Enrolment_update_ratio'].std(skipna=True)
df['z_score']=((df['Enrolment_update_ratio']-mean_update_ratio)/std_ratio)
anamolies=df.loc[df['z_score'].abs()>2]
anamolies[['state','Enrolment_update_ratio','Total','z_score']]


,state,Enrolment_update_ratio,Total,z_score
22,meghalaya,0.724818,0.259611,5.705033


In [40]:
df['BSI_norm'] = (
    df['bio_age_17+'] /
    (
        df['bio_age_17+'] +
        df['demo_age_17+'] +
        df['enrol_age_18+']
    )
)
df = df.replace([np.inf, -np.inf], np.nan)
df['CII'] = (
    (df['enrol_age_0_5'] + df['enrol_age_5_17']) /
    (df['Total']*1000000).replace(0, np.nan)
)
df['update_intensity'] = (
    (df['Total_update']) /
    df['Total'].replace(0, np.nan)
)

In [42]:
df['AIMS'] = (
    0.4 * (1 - df['update_intensity']) +
    0.3 * (1 - df['BSI_norm']) +
    0.3 * df['CII']
)
df['AIMS_rank'] = df['AIMS'].rank(ascending=False)
top_aims = df.sort_values('AIMS', ascending=False).head(10)
fig = px.bar(
    top_aims,
    x='AIMS',
    y='state',
    orientation='h',
    text=top_aims['AIMS'].round(3),
    color='AIMS',
    color_continuous_scale='Teal',
    title='Top 10 States by Composite Inclusion-Maturity Score (AIMS)'
)
fig.update_layout(
    xaxis_title='AIMS (Higher = Better Balance)',
    yaxis_title='State',
    template='plotly_white',
    height=500
)
fig.update_traces(textposition='outside')
fig.show()

In [43]:
fig = px.scatter(
    df,
    x='AIMS',
    y='Total',
    size='Total',
    color='AIMS',
    hover_name='state',
    color_continuous_scale='Viridis',
    labels={
        'AIMS': 'Composite Inclusion-Maturity Score',
        'Total': 'Total Aadhaar Activity (million)'
    },
    title='Inclusion-Maturity Balance vs Aadhaar Scale'
)
fig.add_vline(
    x=df['AIMS'].median(),
    line_dash='dash',
    annotation_text='Median AIMS'
)

fig.update_layout(
    template='plotly_white',
    height=650
)
fig.show()

In [ ]:
outlier_df = df.copy()
Q1 = outlier_df['CII'].quantile(0.25)
Q3 = outlier_df['CII'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_df['CII_outlier'] = (
    (outlier_df['CII'] < lower_bound) |
    (outlier_df['CII'] > upper_bound)
)
outliers = outlier_df[outlier_df['CII_outlier']]
outliers[['state', 'CII', 'Total']]
fig = px.scatter(
    outlier_df,
    x='Total',
    y='CII',
    color='CII_outlier',
    hover_name='state',
    size='Total',
    color_discrete_map={True: 'red', False: 'gray'},
    labels={
        '': 'Total Aadhaar Activity (Lakhs)',
        'CII': 'Child Inclusion Index'
    },
    title='CII Outliers by Aadhaar Scale'
)
fig.add_hline(y=lower_bound,  line_dash='dash',
    line_color='blue',
    annotation_text='Lower-Bound',
    annotation_position='right',
              )
fig.add_hline(y=upper_bound, line_dash='dash',
              line_color='red',
              annotation_text='Upper-Bound',
              annotation_position='right')

fig.update_layout(
    template='plotly_white',
    height=650
)

fig.show()

In [ ]:
corr_df = df[['CII', 'BSI_norm', 'update_intensity', 'Total']].copy()

corr_matrix = corr_df.corr()
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    title='Correlation Matrix: Aadhaar Structural Metrics'
)

fig.update_layout(
    template='plotly_white',
    height=500
)

fig.show()

In [47]:
fig = px.scatter(
    df,
    x='Total',
    y='update_intensity',
    trendline='ols',
    hover_name='state',
    labels={
        'Toatal': 'Total Aadhaar Activity (million)',
        'update_intensity': 'Update Intensity'
    },
    title='Update Intensity vs Aadhaar Scale'
)
fig.update_layout(
    template='plotly_white',
    height=600
)
fig.show()

In [48]:
fig = px.scatter(
    df,
    x='BSI_norm',
    y='CII',
    trendline='ols',
    hover_name='state',
    labels={
        'BSI_norm': 'Biometric Saturation Index (Normalized)',
        'CII': 'Child Inclusion Index'
    },
    title='Child Inclusion vs Biometric Saturation'
)

fig.update_layout(
    template='plotly_white',
    height=600
)

fig.show()


In [52]:
features = [
    'Total',      
    'update_intensity',   
    'BSI_norm',         
    'CII',                
    'AIMS'               
]
pca_df=df[['state']+features].dropna().copy()
scaler = StandardScaler()
X_scaled=scaler.fit_transform(pca_df[features])
pca=PCA(n_components=2)
principal_components = pca.fit_transform(X_scaled)
pca_df['PC1'] = principal_components[:, 0]
pca_df['PC2'] = principal_components[:, 1]

In [53]:
explained_variance = pca.explained_variance_ratio_
explained_variance

array([0.62468049, 0.23575449])

In [54]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=['PC1', 'PC2']
)
loadings

,PC1,PC2
Total,0.034193,0.792848
update_intensity,-0.521748,0.269232
BSI_norm,-0.376769,-0.478913
CII,0.524925,-0.241063
AIMS,0.555976,0.106950


In [55]:
fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    hover_name='state',
    text='state',
    title='PCA of Aadhaar Systems: Scale vs Maturity',
    labels={
        'PC1': f'PC1 (Scale)  {explained_variance[0]*100:.1f}% variance',
        'PC2': f'PC2 (Maturity/Inclusion)  {explained_variance[1]*100:.1f}% variance'
    }
)
fig.update_traces(textposition='top center')
fig.update_layout(
    template='plotly_white',
    height=650
)
fig.show()

In [57]:
cluster_features = [
    'update_intensity', 
    'BSI_norm',         
    'CII',               
    'AIMS'               
    ]
cluster_df = pca_df[['state'] + cluster_features].dropna().copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df[cluster_features])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_df['cluster'] = kmeans.fit_predict(X_scaled)

centroids = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = (
    pd.DataFrame(centroids, columns=cluster_features)
    .assign(cluster=range(4))
)
def label_cluster(row):
    if row['update_intensity'] < 0.4 and row['CII'] > 0.15:
        return 'Emerging'
    elif row['update_intensity'] < 0.6 and row['BSI_norm'] < 0.6:
        return 'Stabilizing'
    elif row['update_intensity'] > 0.6 and row['BSI_norm'] > 0.7:
        return 'Saturated'
    else:
        return 'Outliers'

cluster_labels = {
    row['cluster']: label_cluster(row)
    for _, row in centroid_df.iterrows()
}

cluster_df['cluster_label'] = cluster_df['cluster'].map(cluster_labels)

c:\Users\alladin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.



In [60]:
viz_df = cluster_df.merge(
    pca_df[['state', 'PC1', 'PC2', 'Total']],
    on='state'
)
fig = px.scatter(
    viz_df,
    x='PC1',
    y='PC2',
    color='cluster_label',
    size='Total',
    hover_name='state',
    title='State Clusters in Aadhaar System Space',
    labels={
        'PC1': 'Scale (PCA)',
        'PC2': 'Maturity & Inclusion (PCA)'
    }
)
fig.update_layout(
    template='plotly_white',
    height=650,
    legend_title_text='Cluster Type'
)
fig.show()